In [1]:
import pandas as pd
import json

PATH_RAW = '../data/raw/'

account= pd.read_csv(f'{PATH_RAW}ravenstack_accounts.csv')
churn_events= pd.read_csv(f'{PATH_RAW}ravenstack_churn_events.csv')
feature_usage= pd.read_csv(f'{PATH_RAW}ravenstack_feature_usage.csv')
subscriptions= pd.read_csv(f'{PATH_RAW}ravenstack_subscriptions.csv')
support_tickets= pd.read_csv(f'{PATH_RAW}ravenstack_support_tickets.csv')




In [2]:
##Her tablo için kontroller
account.shape
account.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   account_id       500 non-null    str  
 1   account_name     500 non-null    str  
 2   industry         500 non-null    str  
 3   country          500 non-null    str  
 4   signup_date      500 non-null    str  
 5   referral_source  500 non-null    str  
 6   plan_tier        500 non-null    str  
 7   seats            500 non-null    int64
 8   is_trial         500 non-null    bool 
 9   churn_flag       500 non-null    bool 
dtypes: bool(2), int64(1), str(7)
memory usage: 32.4 KB


In [3]:
account.head()
account.isna().mean()

account_id         0.0
account_name       0.0
industry           0.0
country            0.0
signup_date        0.0
referral_source    0.0
plan_tier          0.0
seats              0.0
is_trial           0.0
churn_flag         0.0
dtype: float64

In [4]:
print("\n--- Veri Boyutları ---")
print(f"Account shape: {account.shape}")
print(f"Churn shape: {churn_events.shape}")
print(f"Feature shape: {feature_usage.shape}")
print(f"subscriptions shape : {subscriptions.shape}")
print(f"support tickets shape : {support_tickets.shape}")

print("\n--- Account sütunları ---")
print(account.columns.tolist())

print("\n--- Churn sütunları ---")
print(churn_events.columns.tolist())

print("\n--- Feature sütunları ---")
print(feature_usage.columns.tolist())

print("\n--- subscriptions sütunları ---")
print(subscriptions.columns.tolist())

print("\n--- support tickets ---")
print(support_tickets.columns.tolist())

print("\n--- Account infos ---")
print(account.info())

print("\n--- Churn infos ---")
print(churn_events.info())

print("\n--- Feature infos ---")
print(feature_usage.info())

print("\n--- subscriptions infos ---")
print(subscriptions.info())

print("\n--- support tickets infos ---")
print(support_tickets.info())



--- Veri Boyutları ---
Account shape: (500, 10)
Churn shape: (600, 9)
Feature shape: (25000, 8)
subscriptions shape : (5000, 14)
support tickets shape : (2000, 9)

--- Account sütunları ---
['account_id', 'account_name', 'industry', 'country', 'signup_date', 'referral_source', 'plan_tier', 'seats', 'is_trial', 'churn_flag']

--- Churn sütunları ---
['churn_event_id', 'account_id', 'churn_date', 'reason_code', 'refund_amount_usd', 'preceding_upgrade_flag', 'preceding_downgrade_flag', 'is_reactivation', 'feedback_text']

--- Feature sütunları ---
['usage_id', 'subscription_id', 'usage_date', 'feature_name', 'usage_count', 'usage_duration_secs', 'error_count', 'is_beta_feature']

--- subscriptions sütunları ---
['subscription_id', 'account_id', 'start_date', 'end_date', 'plan_tier', 'seats', 'mrr_amount', 'arr_amount', 'is_trial', 'upgrade_flag', 'downgrade_flag', 'churn_flag', 'billing_frequency', 'auto_renew_flag']

--- support tickets ---
['ticket_id', 'account_id', 'submitted_at', 'c

In [5]:
#date sütunlarını str den DATE type a dönüştürelim, min & max'ı kontrol edelim

account['signup_date'] = pd.to_datetime(account['signup_date'], errors='coerce')
churn_events['churn_date'] = pd.to_datetime(churn_events['churn_date'], errors='coerce')
feature_usage['usage_date'] = pd.to_datetime(feature_usage['usage_date'], errors='coerce')
subscriptions['start_date'] = pd.to_datetime(subscriptions['start_date'], errors='coerce')
subscriptions['end_date'] = pd.to_datetime(subscriptions['end_date'], errors='coerce')
support_tickets['submitted_at'] = pd.to_datetime(support_tickets['submitted_at'], errors='coerce')
support_tickets['closed_at'] = pd.to_datetime(support_tickets['closed_at'], errors='coerce')

In [6]:
tables = {
    'accounts': account,
    'subscriptions': subscriptions,
    'feature_usage': feature_usage,
    'support_tickets': support_tickets,
    'churn_events': churn_events,
}

date_summary = pd.concat({
    name: df.select_dtypes(include='datetime').agg(['min', 'max', 'count']).T
    for name, df in tables.items()
})
date_summary

min                  max  count
accounts        signup_date   2023-01-02 00:00:00  2024-12-31 00:00:00    500
subscriptions   start_date    2023-01-09 00:00:00  2024-12-31 00:00:00   5000
                end_date      2023-04-05 00:00:00  2024-12-31 00:00:00    486
feature_usage   usage_date    2023-01-01 00:00:00  2024-12-31 00:00:00  25000
support_tickets submitted_at  2023-01-02 00:00:00  2024-12-31 00:00:00   2000
                closed_at     2023-01-03 03:00:00  2024-12-31 19:00:00   2000
churn_events    churn_date    2023-01-25 00:00:00  2024-12-31 00:00:00    600

In [7]:
#submitted_at değerlerinin hepsi gece yarısı (00:00) ama closed_at saat bilgisi de taşıyor (03:00, 19:00 gibi). Yani ticket’ın açılış saati muhtemelen kaydedilmemiş,

In [8]:
#closed_at < submitted_at olan, yani açılmadan kapanmış görünen ticket var mı?
(support_tickets['closed_at'] < support_tickets['submitted_at']).sum()

np.int64(0)

In [9]:
#Grain kontrolü
id_cols = {
    'accounts': 'account_id',
    'subscriptions': 'subscription_id',
    'feature_usage': 'usage_id',
    'support_tickets': 'ticket_id',
    'churn_events': 'churn_event_id',
}

for name, col in id_cols.items():
    df = tables[name]
    print(f"{name:16} rows={len(df):6}  unique {col}={df[col].nunique():6}  is_unique={df[col].is_unique}")

print()
for cols in [['subscription_id', 'usage_date'],
             ['subscription_id', 'usage_date', 'feature_name']]:
    print(f"feature_usage duplicates on {cols}: {feature_usage.duplicated(subset=cols).sum()}")

accounts         rows=   500  unique account_id=   500  is_unique=True
subscriptions    rows=  5000  unique subscription_id=  5000  is_unique=True
feature_usage    rows= 25000  unique usage_id= 24979  is_unique=False
support_tickets  rows=  2000  unique ticket_id=  2000  is_unique=True
churn_events     rows=   600  unique churn_event_id=   600  is_unique=True

feature_usage duplicates on ['subscription_id', 'usage_date']: 78
feature_usage duplicates on ['subscription_id', 'usage_date', 'feature_name']: 3


In [10]:
#Tekrar eden 21 ID var, Eğer satırların tamamı birbirlerinin kopyasıysa bu tekrar eden ID leri sileriz.
feature_usage.duplicated().sum()
#Gösterilecek satır yok, ancak görmek istersem:
feature_usage[feature_usage.duplicated(keep=False)].sort_values('usage_id')

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature


In [11]:
#Sonuç 0, yani tam kopya yok. 21 ID tekrar ediyor ama bu ID’leri taşıyan satırların içeriği farklı. Silinecek bir şey yok.
#Aynı ID'leri taşıyan satırları görelim:
feature_usage[feature_usage.duplicated(subset='usage_id', keep=False)].sort_values('usage_id')


,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
19294,U-0c9318,S-01b2dc,2023-12-11,feature_9,5,1060,3,False
17533,U-0c9318,S-0ffab0,2024-01-30,feature_11,3,1614,0,False
20588,U-13ce5b,S-9b623b,2023-03-26,feature_28,10,2050,1,False
9626,U-13ce5b,S-8b0950,2024-09-10,feature_9,8,824,0,True
18480,U-2103bb,S-7fc49b,2023-04-18,feature_12,9,5022,1,False
10379,U-2103bb,S-ae3270,2024-01-06,feature_3,10,5510,0,False
22,U-25b56c,S-810c27,2024-10-06,feature_20,7,231,0,False
7574,U-25b56c,S-34253c,2023-10-28,feature_20,6,2166,1,False
21376,U-48a4aa,S-93f835,2024-08-24,feature_39,10,4770,0,False
1085,U-48a4aa,S-383ac2,2023-02-12,feature_28,14,7126,0,False


In [13]:
usage_sub = pd.merge(
    feature_usage,
    subscriptions[['subscription_id', 'account_id', 'start_date', 'end_date']],
    left_on='subscription_id',
    right_on='subscription_id',
    how='left')
len(usage_sub)
    

25000

In [17]:
(usage_sub['usage_date'] < usage_sub['start_date']).sum()

np.int64(19142)

In [16]:
(usage_sub['usage_date'] > usage_sub['end_date']).sum()

np.int64(290)